# Harry Potter RAG — Data Prep & Evaluation

This notebook is the **Phase 1** report for the Harry Potter Retrieval-Augmented
Generation (RAG) chatbot. It runs top-to-bottom (Kernel → Restart & Run All) and:

1. Loads and inspects the source PDF (all seven books in one file).
2. Parses it to clean Markdown.
3. Cleans the text (page numbers, artifacts, whitespace/quotes).
4. Chunks it (chapter-aware, tuned to the embedding model) with metadata.
5. Embeds the chunks and persists them to a **Qdrant** vector store on disk.
6. Implements retrieval + a grounded prompt and tests 10 sample questions.
7. Evaluates the full pipeline and tabulates grounded vs. hallucinated answers.
8. Exports the collection + `config.json` for the backend to load (not rebuild).

> The heavy lifting lives in `backend/app/services/ingest.py`,
> `retrieval.py`, and `generation.py`, which the **FastAPI backend uses too** —
> so this report and production run on identical logic.

In [ ]:
# --- Setup: make the backend package importable and configure logging ---
import sys, json, logging
from pathlib import Path

# Locate the project root (the folder that contains `backend/`).
here = Path.cwd()
PROJECT_ROOT = here if (here / "backend").exists() else here.parent
BACKEND_DIR = PROJECT_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")

from app.core.config import get_settings
from app.services import ingest

settings = get_settings()
PDF_PATH = ingest.DEFAULT_PDF
MD_PATH = ingest.DEFAULT_MARKDOWN

print("Project root :", PROJECT_ROOT)
print("Source PDF   :", PDF_PATH, "(exists:", PDF_PATH.exists(), ")")
print("Vector store :", settings.vector_store_path)
print("Embedding    :", settings.embedding_model)
print("LLM (Groq)   :", settings.groq_model, "| key set:", bool(settings.groq_api_key))

## 5.1 Load & Inspect

Basic sanity checks on the source PDF: page count, whether text extracts cleanly,
an approximate word count, and which of the seven book titles we can detect.

In [ ]:
import fitz  # PyMuPDF (installed as a dependency of pymupdf4llm)

doc = fitz.open(str(PDF_PATH))
print(f"Pages: {doc.page_count}")

# Does text extract cleanly? Peek at a page from the middle of the book.
mid_text = doc[doc.page_count // 2].get_text()
print(f"Text extracts cleanly: {bool(mid_text.strip())}")
print("\n--- Sample (middle page) ---")
print(mid_text[:600])

In [ ]:
# Approximate word count + book-boundary detection over a fast plain-text pass.
raw_plain = "\n".join(doc[i].get_text() for i in range(doc.page_count))
print(f"Approx. word count: {len(raw_plain.split()):,}")

from app.services.ingest import clean_text, detect_books
books_preview = detect_books(clean_text(raw_plain))
print(f"\nDetected {len(books_preview)} book span(s):")
for title, start, end in books_preview:
    print(f"  - {title:45s}  chars {start:>9,} .. {end:>9,}")
doc.close()

## 5.2 Parse PDF → Markdown

We convert the PDF to Markdown with **`pymupdf4llm`** (layout- and heading-aware),
falling back to `pypdf` page-text extraction if needed. The result is saved to
`data/processed/harry_potter.md`.

In [ ]:
raw_md = ingest.pdf_to_markdown(PDF_PATH, MD_PATH)
print(f"Parsed markdown: {len(raw_md):,} characters")
print(f"Saved to: {MD_PATH}")
print("\n--- First 1000 characters ---")
print(raw_md[:1000])

## 5.3 Clean the Text

`clean_text` normalises Unicode (curly quotes, dashes, ligatures), strips
standalone page numbers / "Page N" running footers, removes form-feed artifacts,
and collapses excessive blank lines.

In [ ]:
cleaned = ingest.clean_text(raw_md)
print(f"Before cleaning: {len(raw_md):,} chars")
print(f"After cleaning : {len(cleaned):,} chars  "
      f"({100*(len(raw_md)-len(cleaned))/max(1,len(raw_md)):.1f}% removed)")
print("\n--- Cleaned sample ---")
print(cleaned[5000:6000])

## 5.4 Chunking Strategy

**Why not 500–800 tokens?** The assignment suggests starting at 500–800 tokens,
but the choice must be tuned to the *embedding model*. Our model,
`all-MiniLM-L6-v2`, has a hard **256-token** input window — anything longer is
silently truncated at embedding time, so the tail of a large chunk would never be
represented in its vector. Retrieving a chunk whose vector ignores half its text
hurts recall.

**Decision:** target **~220 tokens** per chunk (safely under 256) with **~15%
overlap (~32 tokens)** to preserve context across cut points. We approximate
tokens as `chars / 4` (a standard English heuristic) to avoid paying the
tokenizer cost during sizing.

**Chapter-aware:** chunks never cross a chapter boundary, so every chunk carries a
clean `book` + `chapter` citation. Each chunk also stores `chunk_index`,
approximate `position` within its book, `char_start`, and a `token_estimate`.

In [ ]:
print(f"TARGET_TOKENS = {ingest.TARGET_TOKENS}, OVERLAP_TOKENS = {ingest.OVERLAP_TOKENS} "
      f"(~{ingest.TARGET_CHARS} / {ingest.OVERLAP_CHARS} chars)")

chunks = ingest.build_chunks(cleaned)
print(f"\nTotal chunks: {len(chunks):,}")

In [ ]:
import pandas as pd

df_chunks = pd.DataFrame([{
    "book": c.book,
    "chapter": c.chapter,
    "tokens_est": c.token_estimate,
    "position": round(c.position, 3),
} for c in chunks])

print("Token-estimate distribution per chunk:")
display(df_chunks["tokens_est"].describe().to_frame().T)

print("\nChunks per book:")
display(df_chunks["book"].value_counts().to_frame("chunks"))

In [ ]:
# Inspect one chunk and its metadata.
sample = chunks[len(chunks) // 2]
print("Book       :", sample.book)
print("Chapter    :", sample.chapter)
print("Chunk index:", sample.chunk_index)
print("Tokens ~   :", sample.token_estimate)
print("Position   :", round(sample.position, 3))
print("\n--- Text ---\n" + sample.text)

## 5.5 Embeddings & Vector Store

We embed every chunk with `all-MiniLM-L6-v2` (384-dim, **normalised** so cosine
similarity equals a dot product) and upsert them into a **Qdrant** collection
persisted to disk at `backend/data/vector_store`. The metadata that produced the
store is written alongside it as `config.json`, so the backend can load without
rebuilding.

> On Apple Silicon this runs on the **MPS** GPU automatically.

In [ ]:
# Build + persist the collection from the exact chunks displayed above.
config = ingest.embed_and_store(chunks, source_pdf=PDF_PATH)
print(json.dumps(config, indent=2))

## 5.6 Retrieval & Prompting

We load the persisted store through the same `Retriever` the backend uses,
retrieve the top-k chunks for each of the 10 required sample questions, and show
the grounded prompt that gets sent to the LLM.

In [ ]:
from app.services.retrieval import Retriever

retriever = Retriever()  # loads config.json + collection + embedding model

SAMPLE_QUESTIONS = [
    "Who is Harry Potter's godfather?",
    "What house was Draco Malfoy sorted into?",
    "What is the name of Harry's owl?",
    "How did Voldemort first lose his powers?",
    "What object was revealed to be a Horcrux in the Chamber of Secrets?",
    "Who teaches Defense Against the Dark Arts in Harry's third year?",
    "What is the incantation for the Patronus Charm?",
    "Who kills Dumbledore, and in which book?",
    "What house does the Sorting Hat almost put Harry in?",
    "What are the three Deathly Hallows?",
]

# Quick look at retrieval quality for the first three questions.
for q in SAMPLE_QUESTIONS[:3]:
    hits = retriever.retrieve(q, k=settings.top_k)
    print(f"\nQ: {q}")
    for h in hits[:3]:
        print(f"   [{h.score:.3f}] {h.book} — {h.chapter}")
        print(f"           {h.text[:110].strip()}...")

In [ ]:
from app.services.generation import build_prompt

demo_hits = retriever.retrieve(SAMPLE_QUESTIONS[0], k=settings.top_k)
print(build_prompt(SAMPLE_QUESTIONS[0], demo_hits, settings.max_context_chars)[:1600])

## 5.7 Evaluation

We run all 10 questions through the **full pipeline** (retrieve → grounding gate →
generate) with the local Ollama model, then tabulate:

| column | meaning |
|---|---|
| **top source** | book + chapter of the highest-scoring retrieved chunk |
| **top score** | cosine similarity of that chunk (grounding signal) |
| **answer** | the model's grounded answer (or a refusal) |
| **correct?** | automated check: does the answer contain the expected fact(s)? |
| **verdict** | grounded ✓ / refused / check ⚠ (answered but expected fact missing) |

The `correct?` check is a keyword heuristic against known canon — it can
under-count correct answers that use different phrasing, so we read the table
critically below.

In [ ]:
from app.services.generation import Generator, REFUSAL

gen = Generator()
ok, msg = gen.ping()
print("Groq:", msg)
assert ok, ("Groq is not available. Add your key to backend/.env:\\n"
            "  GROQ_API_KEY=gsk_your_key_here\\n"
            "(get one free at https://console.groq.com/keys)")

In [ ]:
# Expected facts per question. Each concept is a tuple of acceptable synonyms;
# an answer is "correct" if it contains at least one synonym from every concept.
EXPECTED = [
    [("Sirius",)],                                            # 1 godfather
    [("Slytherin",)],                                         # 2 Draco's house
    [("Hedwig",)],                                            # 3 Harry's owl
    [("rebound", "rebounded", "backfired", "reflected",
      "protect", "sacrifice", "mother", "Lily")],            # 4 lost powers
    [("diary",)],                                             # 5 Chamber Horcrux
    [("Lupin",)],                                             # 6 DADA year 3
    [("Expecto Patronum",)],                                  # 7 Patronus incantation
    [("Snape",), ("Half-Blood Prince", "Half Blood Prince")],# 8 kills Dumbledore + book
    [("Slytherin",)],                                         # 9 Sorting Hat almost
    [("Elder Wand", "Elder"), ("Cloak",),
     ("Resurrection Stone", "Resurrection")],                # 10 three Hallows
]

def is_correct(answer: str, expected) -> bool:
    a = answer.lower()
    return all(any(syn.lower() in a for syn in concept) for concept in expected)

rows = []
for i, q in enumerate(SAMPLE_QUESTIONS):
    hits = retriever.retrieve(q, k=settings.top_k)
    relevant = [h for h in hits if h.score >= settings.min_relevance_score]
    answer = gen.generate(q, relevant) if relevant else REFUSAL
    top = hits[0] if hits else None
    refused = REFUSAL.rstrip(".").lower() in answer.lower()
    correct = is_correct(answer, EXPECTED[i])
    verdict = "refused" if refused else ("grounded ✓" if correct else "check ⚠")
    rows.append({
        "#": i + 1,
        "question": q,
        "top source": f"{top.book} — {top.chapter}" if top else "—",
        "top score": round(top.score, 3) if top else 0.0,
        "answer": answer.replace("\\n", " "),
        "correct?": "✓" if correct else "✗",
        "verdict": verdict,
    })

df_eval = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 90)
display(df_eval)

n_correct = sum(r["correct?"] == "✓" for r in rows)
print(f"\nAutomated correctness: {n_correct}/{len(rows)}")

In [ ]:
# Save the evaluation table to Markdown so the README can embed it.
eval_md_path = PROJECT_ROOT / "data" / "processed" / "eval_results.md"
try:
    table_md = df_eval.to_markdown(index=False)
except Exception:
    # Fallback if `tabulate` is not installed.
    table_md = df_eval.to_csv(index=False)
eval_md_path.write_text(table_md, encoding="utf-8")
print("Wrote", eval_md_path)
print(table_md[:1500])

### Failure cases & mitigations

Reading the table above, the recurring failure modes and how we mitigate them:

- **Phrasing mismatches, not real errors.** The `correct?` column is a keyword
  check, so a right answer worded differently (e.g. Q4 "the curse rebounded on
  him thanks to Lily's protection" vs. our synonym list) can show `check ⚠`.
  These are evaluation artifacts, not hallucinations — the verdict column plus a
  manual read separates them. *Mitigation:* an LLM-as-judge grader would score
  semantics instead of keywords.

- **UK vs. US naming.** "Philosopher's Stone" vs. "Sorcerer's Stone" and
  "Defence" vs. "Defense" can shift retrieval. *Mitigation:* the ingest and
  router layers accept both spellings.

- **Multi-hop / whole-plot questions** (e.g. "how did Voldemort *first* lose his
  powers") may need context spread across chapters that a single top-k window
  misses. *Mitigation:* modest `top_k` (5) plus chunk overlap; raising `top_k`
  or adding re-ranking would help further.

- **The grounding gate prevents the worst failure — confident hallucination.**
  Any question whose best chunk falls below `MIN_RELEVANCE_SCORE` is refused with
  "I don't know" rather than answered from the model's parametric memory. This is
  the behaviour the assignment asks for and the reason out-of-scope questions do
  not produce fabricated book facts.

## 5.8 Export

The collection and `config.json` are already persisted under
`backend/data/vector_store/` by step 5.5. Here we verify the artifacts the
backend will load (without rebuilding).

In [ ]:
retriever.close()  # release the on-disk Qdrant lock so the backend can open it

store = settings.vector_store_path
print("Vector store contents:")
for p in sorted(store.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(store)}  ({p.stat().st_size:,} bytes)")

print("\nconfig.json:")
print(settings.config_json_path.read_text())
print("\n✅ Phase 1 complete — the backend can now load this store read-only.")